In [ ]:
import os
N_cores = 1
os.environ["OMP_NUM_THREADS"] = str(N_cores)        # export OMP_NUM_THREADS=4
os.environ["OPENBLAS_NUM_THREADS"] = str(N_cores)   # export OPENBLAS_NUM_THREADS=4
os.environ["MKL_NUM_THREADS"] = str(N_cores)        # export MKL_NUM_THREADS=6
os.environ["VECLIB_MAXIMUM_THREADS"] = str(N_cores) # export VECLIB_MAXIMUM_THREADS=4
os.environ["NUMEXPR_NUM_THREADS"] = str(N_cores)    # export NUMEXPR_NUM_THREADS=6

import numpy as np
import matplotlib.pyplot as plt

import sys

#from ham_creation import create_hex_ham
from lat_creation import get_positions_graphene
from core import DOS_sparse, frequency_analysis, load_data
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
# Benchmarking
from time import time
from datetime import timedelta

hbar = 0.6582


# Matplotlib stuff
plt.rcParams.update({'font.size': 11})  # Match REVTeX default 10pt
plt.rcParams['pdf.fonttype'] = 42  # Save text as TrueType (editable) instead of paths
plt.rcParams['ps.fonttype'] = 42   # Same for PostScript
plt.rcParams['text.usetex'] = True  # Keep LaTeX rendering
plt.rcParams['font.family'] = 'serif'
#plt.rcParams['font.serif'] = ['Times New Roman']

plt.rcParams.update({
    'font.size': 11,              # Tamaño base general (afecta a textos libres)
    'figure.titlesize': 18,       # Título principal de la figura (suptitle)
    'axes.titlesize': 12,         # Título de cada gráfico (title)
    'axes.labelsize': 12,         # Etiquetas de los ejes (xlabel e ylabel)
    'xtick.labelsize': 11,        # Números/marcas del eje X
    'ytick.labelsize': 11,        # Números/marcas del eje Y
    'legend.fontsize': 11,        # Texto de los elementos de la leyenda
    'legend.title_fontsize': 11   # Título de la leyenda
})

# Configuración exclusiva para grosores y tamaños de elementos visuales
plt.rcParams.update({
    # 1. Líneas y Marcadores (El contenido de tus gráficos)
    'lines.linewidth': 3,          # Grosor de las líneas de los gráficos
    'lines.markersize': 10,           # Tamaño de los puntos/marcadores
    'lines.markeredgewidth': 1.5,    # Grosor del borde de los marcadores

    # 2. Gráficos de barras y formas (Patches)
    #'patch.linewidth': 1.2,          # Grosor del borde de las barras (plt.bar) o rectángulos

    # 3. El marco del gráfico (Ejes / Spines)
    #'axes.linewidth': 1.5,           # Grosor de la línea del recuadro exterior del gráfico

    # 4. La cuadrícula (Grid)
    'grid.linewidth': 0.8,           # Grosor de las líneas de la cuadrícula de fondo

    # 5. Las pequeñas marcas de los ejes (Ticks)
    'xtick.major.size': 6,           # Qué tan largas son las marcas del eje X
    'xtick.major.width': 1.5,        # Qué tan gordas son las marcas del eje X
    'ytick.major.size': 6,           # Qué tan largas son las marcas del eje Y
    'ytick.major.width': 1.5         # Qué tan gordas son las marcas del eje Y
})


In [ ]:
# ------------------------------------------------------------------------------
#### PARAMETERS OF THE MODEL
out_file_loc = ''
## PHYSICAL
# Type of light               
modifier_id = 'linear'
# Hamiltonian type
type_ham = 'hbn'
# Parameters of the ham (only read if hbn)
ham_params = 0.5
# Energy in pulse                        
E = 1.1
# Temperature                       
Temp = 1e-9
# Chemical potential
mu = 0.01
# Intensity param     (no units)
gamma_list = np.linspace(0.000, 0.025, 6)                   

## SIMULATION
# Size of hamiltonian
N_pot = 17
N = 2**N_pot
# Amount of periods to be simulated
n_periods = 100
# Simulation steps per period
steps_per_T = 1000
# Amount of measures per period
meas_per_T = 16
N_measures = meas_per_T*n_periods
# Amount of random vectors used in calculation
N_random_vector = 1
# Momenta
M = int(np.sqrt(N))
#M = 362

## RESULT ANALYSIS
# Range of searching the maximim frequency (in period^-1 units)
range_search = 1

## CALCULATED PARAMS
# Parameters of the laser
w = E/jcl.hbar_fs 
T = 2*np.pi/w    
t_vec = np.linspace(0,n_periods*T , steps_per_T*n_periods)   
# Amount of half multiples of E where the occupation is obtained
hE_reps = 2
# Broadening in the energies
t_vec_measures = np.linspace(0, n_periods*T, N_measures)






In [ ]:
## RESULT ANALYSIS
# Range of searching the maximim frequency (in period^-1 units)
hE_reps = 2
## CALCULATED PARAMS


char_freq = np.zeros((len(gamma_list), 2*hE_reps + 1))

timeframe_intensity_list = [(0.000, 100, 4, 1.0),
                            (0.005, 500, 4, 1.0),
                            (0.010, 200, 8, 1.0),
                            (0.015, 150, 8, 1.0),
                            (0.020, 100, 16, 1.0),
                            (0.025, 100, 16, 1.0),
                            (0.030, 80, 24, 1.0),
                            (0.035, 80, 24, 1.0),
                            (0.040, 50, 32, 1.0),
                            (0.045, 50, 32, 1.0),
                            (0.050, 50, 32, 1.0)]

for (g, gamma) in enumerate(gamma_list):
    if M == 0:
        M = int(np.sqrt(N))
    n_periods, meas_per_T, E = timeframe_intensity_list[g][1:]

    # Parameters of the laser
    w = E/jcl.hbar_fs 
    T = 2*np.pi/w    
    t_vec = np.linspace(0,n_periods*T , steps_per_T*n_periods)   
    # Amount of half multiples of E where the occupation is obtained
    
    # Broadening in the energies
    N_measures = meas_per_T*n_periods
    t_vec_measures = np.linspace(0, n_periods*T, N_measures)
    hE_list = [hE*E/2 for hE in range(-hE_reps, hE_reps+1)]
    color_list = ['olivedrab', 'red', 'orange', 'blue', 'darkviolet']
    # Names of file and info on graphs
    folder_name = f'{modifier_id}/G={gamma:.3f}_E={float(E)}_Temp={Temp}_mu={mu:.2f}/N={N_pot}_M={M}_R={N_random_vector}_nT={n_periods}_measT={meas_per_T}_stT={steps_per_T}'
    fig_title_info = f'$N={{{2**N_pot}}}$, E={E} eV, T={Temp} K, $\\mu$={mu} eV, $\\Gamma$={gamma}, M={M}'

    # Loading the info in the .npy files
    EF_list, n_E_list, dos_list, dosn_list = load_data(modifier_id, N_pot, E, Temp, mu, gamma, 
                        M, N_random_vector, n_periods, meas_per_T, steps_per_T, type_ham, ham_params, R=None)

    # Getting the characteristic period of each system
    occ_drop_list, fourier_occ, freq, char_freq[g], max_freq_ind = frequency_analysis(EF_list, dosn_list, hE_list, t_vec_measures, T)
    # --------------------------------------------------------------------------
    # OCCUPATION(time) GRAPH
    # General figure to contain all important data
    fig, ax = plt.subplots()
    reescale = np.max(occ_drop_list[3]) / np.max(occ_drop_list[4]) / 2
    ax.plot(np.array(t_vec_measures)/T, occ_drop_list[4]*reescale, c='darkviolet', 
            marker='.', ls='--', label=f"$E=1\\hbar\\omega$ $\\cdot$ {reescale:.3f}")
    ax.plot(np.array(t_vec_measures)/T, occ_drop_list[3], c='blue', 
            marker='.', ls='--', label=f"$E=0.5\\hbar\\omega$ eV")
    ax.plot(np.array(t_vec_measures)/T, occ_drop_list[2], c='orange', 
            marker='.', ls='--', label=f"$E=0.0\\hbar\\omega$ eV")
    ax.set_xlabel('Time (Periods)')
    ax.set_ylabel('$n(t)$')
    ax.set_title(fr'Occupation')
    fig.suptitle(fig_title_info)
    ax.legend()

    # --------------------------------------------------------------------------
    # FREQUENCY ANALYSIS OF THE RESULTS
    # General figure
    max_freq = min(3, freq[-1]*T/(2*np.pi))
    props = dict(boxstyle='round', facecolor='white', edgecolor='grey', alpha=0.8)

    fig, ax = plt.subplots()
    ax.plot(freq*T/(2*np.pi), fourier_occ[2], c='orange', marker='.', ls='--', 
            label=f'$E = 0\\hbar\\omega$')
    ax.plot(freq*T/(2*np.pi), fourier_occ[3], c='blue', marker='.', ls='--', 
            label=f'$E = 0.5\\hbar\\omega$')
    ax.plot(freq*T/(2*np.pi), fourier_occ[4], c='darkviolet', marker='.', ls='--', 
            label=f'$E = 1\\hbar\\omega$')
    # Markers of max frequencies
    ax.scatter(freq[max_freq_ind[2]]*T/(2*np.pi), fourier_occ[2, max_freq_ind[2]], color='red',
            marker='*', zorder=2)
    ax.scatter(freq[max_freq_ind[3]]*T/(2*np.pi), fourier_occ[3, max_freq_ind[3]], color='cyan',
            marker='*', zorder=2)
    ax.scatter(freq[max_freq_ind[4]]*T/(2*np.pi), fourier_occ[4, max_freq_ind[4]], color='magenta',
            marker='*', zorder=2)
    ax.vlines([range_search], 0, np.max(fourier_occ), ls='-.', color='gray')
    ax.set_xlim(0, max_freq)
    wc_text = f'$\\omega_c = {char_freq[g, 2]:.6f}$ fs$^{{-1}}$\n$\\omega_c = {char_freq[g, 3]:.6f}$ fs$^{{-1}}$\n$\\omega_c = {char_freq[g, 4]:.6f}$ fs$^{{-1}}$'
    ax.text(0.72, 0.98, wc_text, transform=ax.transAxes,
            verticalalignment='top', bbox=props)
    ax.legend(loc=(0.47, 0.7815), labelspacing=0.8, edgecolor='grey')
    ax.set_xlabel('Normal Frequency (period$^{-1}$)')
    ax.set_ylabel('Amplitude')
    ax.set_title('FFT')
    fig.suptitle(fig_title_info)
    
    
# Final plot of the gammas
fig, ax = plt.subplots()
#ax.plot(gamma_list, char_freq[:,2], ls='--', c=color_list[2], marker='.',
#        label=f'$E = \\mu$ eV')
ax.plot(gamma_list, char_freq[:,3], ls='--', c='blue', marker='.',
        label=f'$E = 0.5\\hbar\\omega$')
ax.plot(gamma_list, char_freq[:,4], ls='--', c='darkviolet', marker='.',
        label=f'$E =1\\hbar\\omega$')
ax.set_xlabel(f'Intensity $\\Gamma$')
ax.set_ylabel('Angular freq. (fs$^{-1}$)')
ax.set_title(f'Characteristic frequencies for different intensities')
ax.legend()
ax.set_xticks(gamma_list)
fig.suptitle(f'$N={{{2**N_pot}}}$, $\\hbar\\omega$={E} eV, Temp={Temp} K, $\\mu$={mu} eV')


In [ ]:
# Final plot of the gammas
fig, ax = plt.subplots()
#ax.plot(gamma_list, char_freq[:,2], ls='--', c=color_list[2], marker='.',
#        label=f'$E = \\mu$ eV')
ax.plot(gamma_list, char_freq[:,3], ls='--', c=color_list[3], marker='.',
        label=f'$E = 0.5\\hbar\\omega$')
ax.plot(gamma_list, char_freq[:,4], ls='--', c=color_list[4], marker='.',
        label=f'$E =1\\hbar\\omega$')
ax.set_xlabel(f'Intensity $\\Gamma$')
ax.set_ylabel('Angular frequency (las. period$^{-1}$)')
ax.set_title(f'Characteristic frequencies for different intensities')
ax.legend()
ax.set_xticks(gamma_list)
fig.suptitle(f'$N={{{2**N_pot}}}$, $\\hbar\\omega$={E} eV, Temp={Temp} K, $\\mu$={mu} eV')



In [ ]:

#%% Fitting to see if a relation is even possible
# Importing the results from Floquet
model_select = 'real'
gamma_fl = np.load(f'/home/eperez/Code/Floquet_tfm/Outr/GAP_hw={E}/gamma_list.npy')
gap0_fl = np.load(f'/home/eperez/Code/Floquet_tfm/Outr/GAP_hw={E}/gap0_{model_select}.npy')
gap1_fl = np.load(f'/home/eperez/Code/Floquet_tfm/Outr/GAP_hw={E}/gap1_{model_select}.npy')
gap2_fl = np.load(f'/home/eperez/Code/Floquet_tfm/Outr/GAP_hw={E}/gap2_{model_select}.npy')

# Seeing which indexes are used in the other results
used_inds = []
for (i, g_fl) in enumerate(gamma_fl):
    if g_fl in gamma_list:
        used_inds.append(i)

# Applying the results
gamma_fl = gamma_fl[used_inds]
gap0_fl = gap0_fl[used_inds]
gap1_fl = gap1_fl[used_inds]
gap2_fl = gap2_fl[used_inds]

# Fitting of the curve
from scipy.stats import linregress
# GAP1 REGRESSION
lin_reg1 = linregress(gap1_fl, char_freq[:,3]*jcl.hbar_fs)
slope1 = lin_reg1.slope
intercept1 = lin_reg1.intercept
R2_1 = lin_reg1.rvalue**2
test_gaps1 = np.linspace(np.min(gap1_fl), np.max(gap1_fl), 100)
lin_predict1 = slope1*test_gaps1 + intercept1

# GAP2 REGRESSION
lin_reg2 = linregress(gap2_fl, char_freq[:,4]*jcl.hbar_fs)
slope2 = lin_reg2.slope
intercept2 = lin_reg2.intercept
R2_2 = lin_reg2.rvalue**2
test_gaps2 = np.linspace(np.min(gap2_fl), np.max(gap2_fl), 100)
lin_predict2 = slope2*test_gaps2 + intercept2

# Graph of the results
fig, ax = plt.subplots(dpi=200)
ax.plot(gap1_fl, char_freq[:,3]*jcl.hbar_fs, ls='--', marker='.', color='blue', label='$\\Delta_1$ points')
ax.plot(test_gaps1, lin_predict1, color='cyan', label=f'$y={slope1:.3f}x+{intercept1:.3f}, R^2={R2_1:.3f} $')

ax.plot(gap2_fl, char_freq[:,4]*jcl.hbar_fs, ls='--', marker='.', color='orange', label='$\\Delta_2$ points')
ax.plot(test_gaps2, lin_predict2, color='red', label=f'$y={slope2:.3f}x+{intercept2:.3f}, R^2={R2_2:.3f} $')
ax.set_xlabel('Gap size (eV)')
ax.set_ylabel('$\\hbar \\omega_{char}$ (eV)')
ax.set_title('Comparison Floquet vs NEQ')
ax.legend()

In [ ]:
#%% Trying Rabi stuff to see if theoretical results make sense using gap
t = -2.7
Phi0 = 2*np.pi*jcl.hbar_fs
A0 = gamma_list*Phi0/(2*jcl.a_cc*np.sqrt(3))
vf = 3*jcl.a_cc*abs(t)/(2*jcl.hbar_fs)
w1 = 2*vf*A0/jcl.hbar_fs
w1 = np.sqrt(3)*t*gamma_list*np.pi/jcl.hbar_fs

rep = 1
delta_w = (1-rep)*w
#delta_w = w_laser - gap1_fl/jcl.hbar_fs
rabi_freq = np.sqrt(delta_w**2 + w1**2)
#rabi_freq = 2*vf*A0/jcl.hbar_fs*np.sqrt(1+E**2/(vf**2*A0**2))
w**2*jcl.hbar_fs/(vf*A0)


fig, ax = plt.subplots(dpi=200)
ax.plot(gamma_list, char_freq[:,3], ls='--', c=color_list[3], marker='.',
        label=f'$\\omega_c$ (NUMERICAL RESULTS)')
ax.plot(gamma_list, rabi_freq, ls='--', c='red', marker='.',
        label=f'$\\Omega$')
ax.plot(gamma_list, rabi_freq/2, ls='--', c='orange', marker='.',
        label=f'$\\Omega/2$')
ax.set_title('Characteristic Frequencies in $E=0.5\\hbar\\omega$')
ax.set_xlabel('Intensity parameter $\\Gamma$')
ax.set_ylabel('Ang. frequency')
ax.legend()

#np.save('Out/char_freqs.npy', char_freq[:,3])